In [8]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [5]:
# Create all gold layer tables
DB_GOLD   = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")



tables = [
    "dim_channel",
    "dim_claim_type",
    "dim_member_segment",
    "dim_product_line",
    "dim_providers",
    "dim_region",
    "dm_claims_experience",
    "dm_member_value",
    "dm_policy_retention",
    "dq_monitoring",
    "fact_claims",
    "fact_members",
    "fact_policies",
    "ft_claims_risk",
    "ft_claims_risk_split",
    "ft_policy_churn",
    "ft_policy_churn_split",
    "ml_monitoring",
    "ml_monitoring_view",
    "scored_policy_churn",
    "star_claims",
    "star_members",
    "star_policies",
    "vw_claims_experience_kpi",
    "vw_member_profile",
    "vw_member_value_kpi",
    "vw_ml_claims_risk_features",
    "vw_ml_policy_churn_features",
    "vw_policy_portfolio",
    "vw_policy_retention_kpi",
    "vw_scored_claims_fraud",
    "vw_scored_claims_high_cost",
    "vw_scored_policy_churn"
]

for table in tables:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DB_GOLD}.{table}
        USING DELTA
        LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/{table}'
    """)

print(f"Created {len(tables)} tables in {DB_GOLD} database")

Created 33 tables in bupa_gold database


In [4]:
# Print schemas from all tables in the gold layer
for table_name in tables:
    try:
        df = spark.table(f"{DB_GOLD}.{table_name}")
        print(f"\n{'='*60}")
        print(f"Schema for {DB_GOLD}.{table_name}")
        print(f"{'='*60}")
        df.printSchema()
    except Exception as e:
        print(f"Error reading {DB_GOLD}.{table_name}: {str(e)}")


Schema for bupa_gold.dim_channel
root
 |-- Channel_Code: string (nullable = true)
 |-- Channel_Name: string (nullable = true)


Schema for bupa_gold.dim_claim_type
root
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Type: string (nullable = true)


Schema for bupa_gold.dim_member_segment
root
 |-- Member_Segment_Key: long (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string (nullable = true)
 |-- Chronic_Flag: integer (nullable = true)
 |-- Chronic_Group: string (nullable = true)
 |-- Employment_Group: string (nullable = true)
 |-- Region_Group: string (nullable = true)


Schema for bupa_gold.dim_product_line
root
 |-- Product_Line_Code: string (nullable = true)
 |-- Product_Line: string (nullable = true)


Schema for bupa_gold.dim_member_segment
root
 |-- Member_Segment_Key: long (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string

In [ ]:
import json

def dump_schema(df, path):
    schema = [
        {"name": f.name, "type": f.dataType.simpleString(), "nullable": f.nullable}
        for f in df.schema.fields
    ]
    with open(path, "w") as f:
        json.dump(schema, f, indent=2)

dump_schema(spark.table("bupa_gold.fact_claims"), "/Users/manojrammopati/Public/Projects/bupa_insurance_project/schemas/gold/fact_claims.json")
dump_schema(spark.table("bupa_gold.star_policies"), "schemas/gold/star_policies.json")
dump_schema(spark.table("bupa_gold.star_claims"), "schemas/gold/star_claims.json")
dump_schema(spark.table("bupa_gold.ft_policy_churn"), "schemas/gold/ft_policy_churn.json")
dump_schema(spark.table("bupa_gold.ft_claims_risk"), "schemas/gold/ft_claims_risk.json")
dump_schema(spark.table("bupa_gold.scored_policy_churn"), "schemas/gold/scored_policy_churn.json")


In [6]:
import json
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

DB = os.getenv("DB_GOLD", "bupa_gold")
BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")
OUT = Path("schemas/gold")
OUT.mkdir(parents=True, exist_ok=True)

TABLES = [
  "dim_channel","dim_claim_type","dim_member_segment","dim_product_line","dim_providers","dim_region",
  "fact_claims","fact_members","fact_policies",
  "dm_claims_experience","dm_member_value","dm_policy_retention",
  "star_claims","star_members","star_policies",
  "ft_policy_churn","ft_policy_churn_split","ft_claims_risk","ft_claims_risk_split",
  "dq_monitoring","ml_monitoring","ml_monitoring_view",
  "scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("schema-snapshot")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

for t in TABLES:
    path = f"{BASE}/{t}"
    df = spark.read.format("delta").load(path)
    schema_json = json.loads(df.schema.json())["fields"]
    with open(OUT / f"{t}.json", "w") as f:
        json.dump(schema_json, f, indent=2)
    print("Wrote", OUT / f"{t}.json")


25/12/15 16:39:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Wrote schemas/gold/dim_channel.json
Wrote schemas/gold/dim_claim_type.json
Wrote schemas/gold/dim_member_segment.json
Wrote schemas/gold/dim_member_segment.json
Wrote schemas/gold/dim_product_line.json
Wrote schemas/gold/dim_product_line.json
Wrote schemas/gold/dim_providers.json
Wrote schemas/gold/dim_region.json
Wrote schemas/gold/fact_claims.json
Wrote schemas/gold/dim_providers.json
Wrote schemas/gold/dim_region.json
Wrote schemas/gold/fact_claims.json
Wrote schemas/gold/fact_members.json
Wrote schemas/gold/fact_policies.json
Wrote schemas/gold/dm_claims_experience.json
Wrote schemas/gold/dm_member_value.json
Wrote schemas/gold/dm_policy_retention.json
Wrote schemas/gold/fact_members.json
Wrote schemas/gold/fact_policies.json
Wrote schemas/gold/dm_claims_experience.json
Wrote schemas/gold/dm_member_value.json
Wrote schemas/gold/dm_policy_retention.json
Wrote schemas/gold/star_claims.json
Wrote schemas/gold/star_members.json
Wrote schemas/gold/star_policies.json
Wrote schemas/gold/f

In [7]:
import json
import os
from pathlib import Path

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")
OUT = Path("schemas/gold")
OUT.mkdir(parents=True, exist_ok=True)

TABLES = [
    "dim_channel","dim_claim_type","dim_member_segment","dim_product_line","dim_providers","dim_region",
    "dm_claims_experience","dm_member_value","dm_policy_retention",
    "dq_monitoring",
    "fact_claims","fact_members","fact_policies",
    "ft_claims_risk","ft_claims_risk_split","ft_policy_churn","ft_policy_churn_split",
    "ml_monitoring","ml_monitoring_view",
    "scored_policy_churn",
    "star_claims","star_members","star_policies",
    # “vw_*” folders exist locally → snapshot them too
    "vw_claims_experience_kpi","vw_member_profile","vw_member_value_kpi",
    "vw_ml_claims_risk_features","vw_ml_policy_churn_features",
    "vw_policy_portfolio","vw_policy_retention_kpi",
    "vw_scored_claims_fraud","vw_scored_claims_high_cost","vw_scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("schema-snapshot")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

for t in TABLES:
    path = f"{BASE}/{t}"
    df = spark.read.format("delta").load(path)
    fields = json.loads(df.schema.json())["fields"]
    out_path = OUT / f"{t}.json"
    out_path.write_text(json.dumps(fields, indent=2))
    print("Wrote", out_path)

spark.stop()


Wrote schemas/gold/dim_channel.json
Wrote schemas/gold/dim_claim_type.json
Wrote schemas/gold/dim_member_segment.json
Wrote schemas/gold/dim_product_line.json
Wrote schemas/gold/dim_providers.json
Wrote schemas/gold/dim_region.json
Wrote schemas/gold/dm_claims_experience.json
Wrote schemas/gold/dm_member_value.json
Wrote schemas/gold/dm_policy_retention.json
Wrote schemas/gold/dq_monitoring.json
Wrote schemas/gold/fact_claims.json
Wrote schemas/gold/fact_members.json
Wrote schemas/gold/fact_policies.json
Wrote schemas/gold/ft_claims_risk.json
Wrote schemas/gold/ft_claims_risk_split.json
Wrote schemas/gold/ft_policy_churn.json
Wrote schemas/gold/ft_policy_churn_split.json
Wrote schemas/gold/ml_monitoring.json
Wrote schemas/gold/ml_monitoring_view.json
Wrote schemas/gold/scored_policy_churn.json
Wrote schemas/gold/star_claims.json
Wrote schemas/gold/star_members.json
Wrote schemas/gold/star_policies.json
Wrote schemas/gold/star_policies.json
Wrote schemas/gold/vw_claims_experience_kpi.js

In [10]:
# Export sample data from all gold tables to local delta folder
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

BASE = os.getenv("LOCAL_GOLD_BASE", "data/gold")
SAMPLE_OUT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample")
SAMPLE_OUT.mkdir(parents=True, exist_ok=True)

TABLES = [
    "dim_channel", "dim_claim_type", "dim_member_segment", "dim_product_line", "dim_providers", "dim_region",
    "fact_claims", "fact_members", "fact_policies",
    "dm_claims_experience", "dm_member_value", "dm_policy_retention",
    "star_claims", "star_members", "star_policies",
    "ft_policy_churn", "ft_policy_churn_split", "ft_claims_risk", "ft_claims_risk_split",
    "dq_monitoring", "ml_monitoring",
    "scored_policy_churn",
]

builder = (
    SparkSession.builder.master("local[*]").appName("export-gold-samples")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Exporting {len(TABLES)} sample tables to {SAMPLE_OUT}\n")

for t in TABLES:
    src_path = f"{BASE}/{t}"
    dst_path = str(SAMPLE_OUT / t)
    
    try:
        # Read full table
        df = spark.read.format("delta").load(src_path)
        total_rows = df.count()
        
        # Take sample (limit to first 1000 rows or 10% of data, whichever is smaller)
        sample_size = min(1000, max(10, int(total_rows * 0.1)))
        sample_df = df.limit(sample_size)
        
        # Write as Delta table
        sample_df.write.format("delta").mode("overwrite").save(dst_path)
        
        print(f"✅ {t:35s} - {total_rows:,} rows → {sample_size:,} samples exported")
        
    except Exception as e:
        print(f"⚠️ {t:35s} - Error: {str(e)[:60]}")

spark.stop()

print(f"\n{'='*70}")
print(f"✅ Samples exported to: {SAMPLE_OUT}")
print(f"{'='*70}")

Exporting 22 sample tables to /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample

✅ dim_channel                         - 4 rows → 10 samples exported
✅ dim_channel                         - 4 rows → 10 samples exported


✅ dim_claim_type                      - 6 rows → 10 samples exported
✅ dim_member_segment                  - 1,440 rows → 144 samples exported
✅ dim_member_segment                  - 1,440 rows → 144 samples exported


✅ dim_product_line                    - 5 rows → 10 samples exported
✅ dim_providers                       - 5,393 rows → 539 samples exported
✅ dim_providers                       - 5,393 rows → 539 samples exported


✅ dim_region                          - 53 rows → 10 samples exported
✅ fact_claims                         - 558,211 rows → 1,000 samples exported
✅ fact_claims                         - 558,211 rows → 1,000 samples exported
✅ fact_members                        - 381,109 rows → 1,000 samples exported
✅ fact_members                        - 381,109 rows → 1,000 samples exported
✅ fact_policies                       - 381,109 rows → 1,000 samples exported
✅ fact_policies                       - 381,109 rows → 1,000 samples exported


✅ dm_claims_experience                - 558,211 rows → 1,000 samples exported
✅ dm_member_value                     - 381,109 rows → 1,000 samples exported
✅ dm_member_value                     - 381,109 rows → 1,000 samples exported
✅ dm_policy_retention                 - 183 rows → 18 samples exported
✅ dm_policy_retention                 - 183 rows → 18 samples exported
✅ star_claims                         - 558,211 rows → 1,000 samples exported
✅ star_claims                         - 558,211 rows → 1,000 samples exported
✅ star_members                        - 381,109 rows → 1,000 samples exported
✅ star_members                        - 381,109 rows → 1,000 samples exported
✅ star_policies                       - 381,109 rows → 1,000 samples exported
✅ star_policies                       - 381,109 rows → 1,000 samples exported
✅ ft_policy_churn                     - 381,109 rows → 1,000 samples exported
✅ ft_policy_churn                     - 381,109 rows → 1,000 samples exported


✅ ft_policy_churn_split               - 381,109 rows → 1,000 samples exported


✅ ft_claims_risk                      - 558,211 rows → 1,000 samples exported
✅ ft_claims_risk_split                - 558,211 rows → 1,000 samples exported
✅ ft_claims_risk_split                - 558,211 rows → 1,000 samples exported


✅ dq_monitoring                       - 24 rows → 10 samples exported
✅ ml_monitoring                       - 29 rows → 10 samples exported
✅ ml_monitoring                       - 29 rows → 10 samples exported
✅ scored_policy_churn                 - 314,128 rows → 1,000 samples exported
✅ scored_policy_churn                 - 314,128 rows → 1,000 samples exported

✅ Samples exported to: /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample

✅ Samples exported to: /Users/manojrammopati/Public/Projects/bupa_insurance_project/data/gold_sample
